# ESCI Label Audit
End-to-end notebook: data exploration → prompt development → full audit run → results analysis.

In [1]:
import sys
sys.path.insert(0, '..')  # make label_audit/ importable

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

import pandas as pd
import config
from src.data import load_example_products, filter_audit_set

In [2]:
# Resolve paths from the notebook's location — config.__file__ is relative when
# imported via sys.path, so Path(__file__).parent traversal breaks. Override here.
from pathlib import Path

REPO_ROOT = (Path.cwd() / "../..").resolve()
config.DATA_DIR = REPO_ROOT / "shopping_queries_dataset"
config.OUTPUT_DIR = REPO_ROOT / "label_audit" / "output"
config.PROMPT_PATH = REPO_ROOT / "label_audit" / "prompts" / "audit.txt"

## 1. Data Exploration

In [3]:
df = load_example_products(config.DATA_DIR)
print(df.shape)
df.dtypes

(2621288, 14)


example_id              int64
query                     str
query_id                int64
product_id                str
product_locale            str
esci_label                str
small_version           int64
large_version           int64
split                     str
product_title             str
product_description       str
product_bullet_point      str
product_brand             str
product_color             str
dtype: object

In [4]:
# Null rates for product text fields
text_cols = ['product_title', 'product_description', 'product_bullet_point']
df[text_cols].isnull().mean().rename('null_rate')

product_title           0.000000
product_description     0.514508
product_bullet_point    0.139153
Name: null_rate, dtype: float64

In [5]:
TARGET_QUERIES = [
    "aa batteries 100 pack",
    "kodak photo paper 8.5 x 11 glossy",
    "dewalt 8v max cordless screwdriver kit, gyroscopic",
]

# Count of 'E'-labeled rows for the three target queries
audit_set = filter_audit_set(df, TARGET_QUERIES)
print(f"Audit set: {len(audit_set)} rows")
audit_set.groupby('query').size().rename('count')

Audit set: 24 rows


query
aa batteries 100 pack                                  8
dewalt 8v max cordless screwdriver kit, gyroscopic     6
kodak photo paper 8.5 x 11 glossy                     10
Name: count, dtype: int64

In [8]:
# Inspect a sample — read product text before touching the LLM
sample = audit_set.sample(24, random_state=42)[['query', 'product_title', 'product_bullet_point']]
pd.set_option('display.max_colwidth', 200)
sample

,query,product_title,product_bullet_point
8,"dewalt 8v max cordless screwdriver kit, gyroscopic","DEWALT XTREME 12V MAX Cordless Screwdriver, 1/4-Inch, Tool Only (DCF601B)",The cordless screwdriver has 25% more power**\nThe rechargeable screwdriver is 23% shorter**\nBrushless motor of the powered screwdriver is designed for maximum runtime and durability\n1/4-inch qu...
16,kodak photo paper 8.5 x 11 glossy,"Photo Paper, 6.5 mil, Glossy, 8-1/2 x 11, 100 Sheets/Pack",Sold as 100 Sheets/Pack.\nInstant dry.\nNo smearing or smudging.\nWorks on all inkjet printers.
0,aa batteries 100 pack,Energizer Advanced AA Alkaline Bulk Battery - 100 Count,Bulk Packaging
18,kodak photo paper 8.5 x 11 glossy,"Kodak Photo Paper for inkjet printers, Gloss Finish, 7 mil thickness, 50 Sheets, 8.5” x 11” (1213712)","Instant dry. For quality, colorful prints. Affordably priced. No smearing, no smudging. Available in gloss and matte finishes. Universal compatibility.\nInstant dry.\nFor quality, colorful prints...."
11,"dewalt 8v max cordless screwdriver kit, gyroscopic",DEWALT DCB095 8V MAX Battery Charger,Charges all DEWALT 8V MAX Li-Ion batteries\nCharges the battery in 1 hour or less\nDiagnostics with LED indicator
9,"dewalt 8v max cordless screwdriver kit, gyroscopic","ENERTWIST Cordless Screwdriver, 8V Max 10Nm Electric Screwdriver Rechargeable Set with 82 Accessory Kit and Charger in Carrying Case, 21+1 Cluth, Dual Position Handle, LED Light, ET-CS-8","【Powerful 8V Motor & Max 10Nm Torque】Enertwist cordless screwdriver equiped with higher performance 8V motor, delivers improved 88 in.lbs(10Nm) max torque for a wide range of drilling and fastenin..."
13,"dewalt 8v max cordless screwdriver kit, gyroscopic","DEWALT 8V MAX Cordless Screwdriver Kit, Gyroscopic, 1 Battery, Electric (DCF682N1)",The cordless screwdriver features motion activation variable speed and reversing control for precise fastening control\nMotion activated variable speed 0-430 rpm of the rechargeable screwdriver is...
1,aa batteries 100 pack,"IMPECCA AA Batteries, All Purpose Alkaline Batteries (100-Pack) Double A High Performance AA Battery Long Lasting Shelf Life and Leak Resistant 100-Count LR6 - Platinum Series (case Included!)","Packaging may VARY! AA 1.5 volt alkaline batteries, high energy, leak resistant, with long lasting shelf life\nDesigned to provide reliable and lasting performance for both high and low drain devi..."
21,kodak photo paper 8.5 x 11 glossy,"Kodak Photo Paper for inkjet printers, Matte Finish, 7 mil thickness, 100 sheets, 8.5” x 11” (8318164)","Basic matte paper for arts, craft and snapshots\nInstant dry: No smearing or smudging\nGuaranteed to work with any inkjet printer\nWhen a picture on a screen just isn’t enough, trust Kodak, the #1..."
5,aa batteries 100 pack,Energizer AA Max Alkaline E91 Batteries Made in USA - Expiration 12/2024 or Later - 100 Count,Made in USA\nUp to 10 years shelf life\nZero Mercury\nEnergizer MAX AA batteries 50 count\nBulk Packaging


## 2. Prompt Development

Run the LLM against a small subset and inspect raw outputs before committing to a full run.

In [9]:
from src.llm import LLMClient

client = LLMClient(model=config.MODEL_NAME, prompt_path=config.PROMPT_PATH)

# Print the prompt template so you can see and edit it
print(config.PROMPT_PATH.read_text())

You are an expert search relevance assessor for an e-commerce platform.

Your job is to evaluate whether a query-product pair that has been labeled "E" (Exact match) genuinely satisfies the following definition:
  "The item is relevant for the query and satisfies ALL of the query's specifications."

## Decision rules

- If the product information EXPLICITLY CONTRADICTS a specification in the query → the label is INACCURATE.
- If the product does NOT MENTION a specification in the query → leave the label as ACCURATE.
- If the product has ADDITIONAL items or information beyond what was requested → leave the label as ACCURATE.

## Input

Query: {query}

Product Title: {product_title}

Product Description: {product_description}

Product Bullet Points: {product_bullet_point}

## Task

1. Decide whether the "E" label is accurate for this query-product pair.
2. If inaccurate, write a reformulated query that accurately describes this specific product so the "E" label would be correct. The refo

In [10]:
# Test on a handful of rows — iterate on prompts/audit.txt until this looks right
dev_set = audit_set.sample(min(10, len(audit_set)), random_state=0)

dev_results = []
for _, row in dev_set.iterrows():
    result = client.audit_pair(
        query=row['query'],
        product_title=row.get('product_title', ''),
        product_description=row.get('product_description', ''),
        product_bullet_point=row.get('product_bullet_point', ''),
    )
    dev_results.append({'query': row['query'], 'product_title': row['product_title'], **result})

pd.DataFrame(dev_results)

,query,product_title,accurate,reformulated_query
0,"dewalt 8v max cordless screwdriver kit, gyroscopic",DEWALT DCB095 8V MAX Battery Charger,False,dewalt 8v max battery charger
1,"dewalt 8v max cordless screwdriver kit, gyroscopic","DEWALT DCF680N2 8V Max Gyroscopic Screwdriver 2 Battery Kit with DEWALT DWST08201 Tough System Case, Small",True,NaN
2,kodak photo paper 8.5 x 11 glossy,"Kodak Premium Photo Paper for inkjet printers, Gloss Finish, 8.5 mil thickness, 50 Sheets, 8.5” x 11” (8360513),White",True,NaN
3,kodak photo paper 8.5 x 11 glossy,"Kodak photo paper 8.5 x 11 glossy, 50 counts 66 lb - 252 g/m (41159-8360513)",True,NaN
4,kodak photo paper 8.5 x 11 glossy,"Kodak Premium Photo Paper for inkjet printers, Gloss Finish, 8.5 mil thickness, 25 Sheets, 8.5” x 11” (8689283)",True,NaN
5,aa batteries 100 pack,"IMPECCA AA Batteries, All Purpose Alkaline Batteries (100-Pack) Double A High Performance AA Battery Long Lasting Shelf Life and Leak Resistant 100-Count LR6 - Platinum Series (case Included!)",True,NaN
6,"dewalt 8v max cordless screwdriver kit, gyroscopic","DEWALT 8V MAX Cordless Screwdriver Kit, Gyroscopic, 1 Battery, Electric (DCF682N1)",True,NaN
7,kodak photo paper 8.5 x 11 glossy,"KODAK Photo Paper Gloss 8.5""x11"", 25 count, 48lb-180g/m2 weight, 6.5 mil thickness (41161 - 1912369),White",True,NaN
8,kodak photo paper 8.5 x 11 glossy,"Photo Paper, 6.5 mil, Glossy, 8-1/2 x 11, 100 Sheets/Pack",True,NaN
9,"dewalt 8v max cordless screwdriver kit, gyroscopic","DEWALT XTREME 12V MAX Cordless Screwdriver, 1/4-Inch, Tool Only (DCF601B)",False,DEWALT 12V MAX cordless screwdriver kit


## 3. Full Audit Run

In [11]:
from src.auditor import run_audit
from src.output import write_results, print_summary

results = run_audit(audit_set, client)
path = write_results(results, config.OUTPUT_DIR)
print(f"Saved to {path}")

Saved to /Users/hamidbagheri/GitHub/temp/esci-data/label_audit/output/results.csv


## 4. Results Analysis

In [12]:
print_summary(results)

Total pairs audited : 24
Accurate ('E' correct): 17
Mislabeled           : 7

Mislabeled rows:
 query_id product_id                                                reformulated_query
     6014 B00LHSAARW                                              AA batteries 60 pack
     6014 B01B8R6V2E                                            aaa batteries 100 pack
    32814 B07TWK2S22                           DEWALT 12V MAX cordless screwdriver kit
    32814 B0812ZHY5N Enertwist 8V max cordless screwdriver kit with gyroscopic control
    32814 B00EUHAGX0                                     dewalt 8v max battery charger
    58953 B085F42SV6                        kodak photo paper 8.5 x 11 matte 100 count
    58953 B000EZ0CTK                                  Kodak photo paper 8.5 x 11 matte


In [11]:
# Full results table
results

,query_id,product_id,label_accurate,reformulated_query
0,6014,B01G1RYHAO,True,NaN
1,6014,B07FP5DNBG,True,NaN
2,6014,B07F7RH8D4,True,NaN
3,6014,B01B8R6PF2,True,NaN
4,6014,B00LHSAARW,False,aa batteries 60 pack
5,6014,B00KMDL8U6,False,Energizer AA Max Alkaline Batteries 50 count
6,6014,B004SCA15K,True,NaN
7,6014,B01B8R6V2E,False,AAA batteries 100 pack
8,32814,B07TWK2S22,False,dewalt 12v max cordless screwdriver kit
9,32814,B0812ZHY5N,False,dewalt 8v max cordless screwdriver kit


In [12]:
# Mislabeled pairs with original query for context
mislabeled = results[~results['label_accurate']].merge(
    audit_set[['query_id', 'product_id', 'query', 'product_title']],
    on=['query_id', 'product_id'],
)
mislabeled[['query_id', 'product_id', 'query', 'product_title', 'reformulated_query']]

,query_id,product_id,query,product_title,reformulated_query
0,6014,B00LHSAARW,aa batteries 100 pack,"Rayovac AA Alkaline Double A Batteries, 60 Count",aa batteries 60 pack
1,6014,B00KMDL8U6,aa batteries 100 pack,Energizer AA Max Alkaline E91 Batteries Made in USA - Expiration 12/2024 or Later - 100 Count,Energizer AA Max Alkaline Batteries 50 count
2,6014,B01B8R6V2E,aa batteries 100 pack,"Amazon Basics 100 Pack AAA High-Performance Alkaline Batteries, 10-Year Shelf Life, Easy to Open Value Pack",AAA batteries 100 pack
3,32814,B07TWK2S22,"dewalt 8v max cordless screwdriver kit, gyroscopic","DEWALT XTREME 12V MAX Cordless Screwdriver, 1/4-Inch, Tool Only (DCF601B)",dewalt 12v max cordless screwdriver kit
4,32814,B0812ZHY5N,"dewalt 8v max cordless screwdriver kit, gyroscopic","ENERTWIST Cordless Screwdriver, 8V Max 10Nm Electric Screwdriver Rechargeable Set with 82 Accessory Kit and Charger in Carrying Case, 21+1 Cluth, Dual Position Handle, LED Light, ET-CS-8",dewalt 8v max cordless screwdriver kit
5,32814,B00EUHAGX0,"dewalt 8v max cordless screwdriver kit, gyroscopic",DEWALT DCB095 8V MAX Battery Charger,dewalt 8v max battery charger
6,58953,B085F42SV6,kodak photo paper 8.5 x 11 glossy,"Kodak photo paper 8.5 x 11 matte, 100 count 39 lb - 145 g/m (41164-8318164)",kodak photo paper 8.5 x 11 matte
7,58953,B01M0L2WLF,kodak photo paper 8.5 x 11 glossy,"Photo Paper, 6.5 mil, Glossy, 8-1/2 x 11, 100 Sheets/Pack",kodak photo paper 8.5 x 11 glossy 100 sheets
8,58953,B01JB7D4SW,kodak photo paper 8.5 x 11 glossy,"Kodak 8209017 Photo Paper, 6.5 mil, Glossy, 8-1/2 x 11, 100 Sheets/Pack",kodak photo paper 8.5 x 11
9,58953,B000EZTYHG,kodak photo paper 8.5 x 11 glossy,"Kodak Photo Paper for inkjet printers, Gloss Finish, 7 mil thickness, 50 Sheets, 8.5” x 11” (1213712)",kodak photo paper 8.5 x 11 glossy 50 sheets
